In [2]:
import sympy as sp
from sympy.assumptions import Q
#Impostare direttamente le opzioni di pretty printing
sp.init_printing(use_unicode=True, wrap_line=True, num_columns=120)
# simboli
q0, qz, q2 = sp.symbols('q0 qz q2', real=True)
m = sp.symbols('m', positive=True, real=True)
E = sp.symbols('E', real=True, positive=True) 

# definizioni beta± come in Appendice A.3: beta±_i = sqrt(E_i ± m_i)
beta_p = sp.sqrt(E + m)
beta_m = sp.sqrt(E - m)

# Forza le assunzioni: E > m > 0 implica E+m > 0 e E-m > 0
assumptions = Q.positive(E) & Q.positive(m) & Q.gt(E, m)

# Semplifica con assunzioni
beta_p = sp.refine(beta_p, assumptions)
beta_m = sp.refine(beta_m, assumptions)

In [3]:
# Polarisation vectors omega_mu(±), omega_mu(0), omega_mu(t)

###Attenzione alle convenzioni
omega_p = sp.Matrix([0, -1, -sp.I, 0])/sp.sqrt(2)
omega_m = sp.Matrix([0, +1, -sp.I, 0])/sp.sqrt(2)
omega_0 = sp.Matrix([0, 0, 0, 1])
omega_t = sp.Matrix([1, 0, 0, 0])

In [6]:
# Definizione gamma matrices in Dirac basis (Appendice A.12)
#Definizione delle Matrici di Pauli
I2 = sp.eye(2)
sigma1 = sp.Matrix([[0,1],[1,0]])
sigma2 = sp.Matrix([[0,-sp.I],[sp.I,0]])
sigma3 = sp.Matrix([[1,0],[0,-1]])

gamma0 = sp.Matrix(sp.BlockMatrix([[sp.eye(2), sp.zeros(2)],[sp.zeros(2), -sp.eye(2)]]))
gamma1 = sp.Matrix(sp.BlockMatrix([[sp.zeros(2), sigma1],[ -sigma1, sp.zeros(2)]]))
gamma2 = sp.Matrix(sp.BlockMatrix([[sp.zeros(2), sigma2],[ -sigma2, sp.zeros(2)]]))
gamma3 = sp.Matrix(sp.BlockMatrix([[sp.zeros(2), sigma3],[ -sigma3, sp.zeros(2)]]))
gamma5 = sp.Matrix(sp.BlockMatrix([[sp.zeros(2), sp.eye(2)],[sp.eye(2), sp.zeros(2)]]))

# Utility: build 4x4 gamma^mu vector
gamma = [gamma0, gamma1, gamma2, gamma3]

# Definizione spinori u(lambda) e v(lambda) come in Appendix A.3
# Ordering of components as column vectors
u_u_p = sp.Matrix([beta_p, 0, beta_m, 0])    # u(+1/2) in paper
u_u_m = sp.Matrix([0, beta_p, 0, -beta_m])   # u(-1/2)

v_v_p = sp.Matrix([beta_m, 0, -beta_p, 0])   # v(+1/2)
v_v_m = sp.Matrix([0, beta_m, 0, beta_p])  # v(-1/2)

# Definiamo funzioni che restituiscono u(hel) e v(hel) dato hel = +1/2 or -1/2
def u_spinor(hel):
    """Return u spinor for helicity hel (+1/2 or -1/2)."""
    if hel == 1/2 or hel == sp.Rational(1,2):
        return u_u_p
    elif hel == -1/2 or hel == sp.Rational(-1,2):
        return u_u_m
    else:
        raise ValueError("hel must be +1/2 or -1/2")

def v_spinor(hel):
    """Return v spinor for helicity hel (+1/2 or -1/2)."""
    if hel == 1/2 or hel == sp.Rational(1,2):
        return v_v_p
    elif hel == -1/2 or hel == sp.Rational(-1,2):
        return v_v_m
    else:
        raise ValueError("hel must be +1/2 or -1/2")

# Definizione di ubar = u^† gamma0
def ubar(u):
    return (u.conjugate().T * gamma0)

def vbar(v):
    return (v.conjugate().T * gamma0)

# Calcolo delle ampiezze leptoniche LX = ubar Gamma_X v
# Gamma structures: V -> gamma^mu ; A -> gamma^mu gamma5
# Contract with polarisation omega_mu(λX) per richiesta

# funzione che calcola L_V^lambdaX = ubar * (gamma^mu) * v * omega_mu(lambdaX)
def L_vector(u, v, omega):
    # costruzione sommatoria mu con metrica di Minkowski diag(1,-1,-1,-1)
    res = sp.Rational(0)
    g = [1, -1, -1, -1]
    for mu in range(4):
        res += (ubar(u) * gamma[mu] * v)[0] * g[mu] * omega[mu]
    return sp.simplify(res)

# funzione per corrente assiale
def L_axial(u, v, omega):
    res = sp.Rational(0)
    g = [1, -1, -1, -1]
    for mu in range(4):
        res += (ubar(u) * gamma[mu] * gamma5 * v)[0] * g[mu] * omega[mu]
    return sp.simplify(res)


def build_L_matrices(omega=None):
    # hel order: row/col 0 -> -1/2 , 1 -> +1/2
    hel_vals = [sp.Rational(-1,2), sp.Rational(1,2)]
    LV_mat = sp.Matrix([[0,0],[0,0]])
    LA_mat = sp.Matrix([[0,0],[0,0]])

    def pick_omega_from_diff(lam_diff):
        # lam_diff is a rational number: 1 -> omega_+, -1 -> omega_-, 0 -> longitudinal (omega_0)
        if lam_diff == sp.Integer(1):
            return omega_p
        elif lam_diff == sp.Integer(-1):
            return omega_m
        elif lam_diff == sp.Integer(0):
            return omega_0
        else:
            # fallback: time-like (should not occur for helicity differences in this 2x2 setup)
            return omega_t

    for i, lam1 in enumerate(hel_vals):
        for j, lam2 in enumerate(hel_vals):
            lam_diff = lam1 - lam2  # lambda_omega = lambda1 - lambda2
            # choose omega: if caller provided a specific omega, use it for all entries,
            # otherwise pick the polarization according to lam_diff
            chosen_omega = omega if omega is not None else pick_omega_from_diff(lam_diff)
            u = u_spinor(lam1)
            v = v_spinor(lam2)
            LV_mat[i, j] = sp.simplify(L_vector(u, v, chosen_omega))
            LA_mat[i, j] = sp.simplify(L_axial(u, v, chosen_omega))

    return LV_mat, LA_mat

# Costruzione di una sola LV e LA (ognuna con elementi scelti in base a lambda1-lambda2)
LV, LA = build_L_matrices()  # omega=None -> pick_omega_from_diff used per elemento

def force_conjugate_simplification(expr):
    """Forza conjugate(sqrt(x)) = sqrt(x) per espressioni positive"""
    # Sostituisci conjugate(sqrt(E+m)) con sqrt(E+m)
    expr = expr.replace(lambda e: e.func == sp.conjugate and 
                                  len(e.args) == 1 and
                                  e.args[0].func == sp.Pow and 
                                  e.args[0].exp == sp.Rational(1,2),
                       lambda e: e.args[0])
    return sp.simplify(expr)

# Applica la semplificazione elemento per elemento
LV = LV.applyfunc(force_conjugate_simplification)
LA = LA.applyfunc(force_conjugate_simplification)

# Applica refine con assunzioni
LV = sp.refine(LV, assumptions)
LA = sp.refine(LA, assumptions)

q = sp.symbols('q', positive=True, real=True)

# Sostituzioni cinematiche
kinematic_subs = {
    E**2: q**2/4,
    E: sp.sqrt(q**2)/2
}

LV = LV.subs(kinematic_subs)
LA = LA.subs(kinematic_subs)


In [7]:
# Ampiezza di elicità leptonica timelike: L_T = 2m * ubar * gamma5 * v
# Matrice 2x2 con elementi L_T[lambda1, lambda2]

hel_vals = [sp.Rational(-1,2), sp.Rational(1,2)]
LT = sp.Matrix([[0,0],[0,0]])

for i, lam1 in enumerate(hel_vals):
    for j, lam2 in enumerate(hel_vals):
        u = u_spinor(lam1)
        v = v_spinor(lam2)
        LT[i, j] = 2*m * (ubar(u) * gamma5 * v)[0]

# Applica semplificazione
LT = LT.applyfunc(force_conjugate_simplification)
LT = sp.refine(LT, assumptions)
LT = LT.subs(kinematic_subs)
LT = sp.simplify(LT)

In [8]:
LLeft = LV - LA
LRight = LV + LA

# Visualizzazione LaTeX direttamente in Jupyter (senza salvare)
from IPython.display import display, Math

display(Math(r'\mathbf{L}_{\text{Vector}} = ' + sp.latex(LV)))
display(Math(r'\mathbf{L}_{\text{Axial}} = ' + sp.latex(LA)))

display(Math(r'\mathbf{L}_{\text{Left}} = ' + sp.latex(LLeft)))
display(Math(r'\mathbf{L}_{\text{Right}} = ' + sp.latex(LRight)))

display(Math(r'\mathbf{L}_{\text{Timelike}} = ' + sp.latex(LT)))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>